# D3Net Fine-Tuning on Google Colab (SIDL Dataset)

이 노트북은 Google Colab 환경에서 **D3Net** 모델을 **SIDL Dataset**으로 학습(Fine-Tuning) 및 평가하기 위한 실행 가이드라인입니다.

### ⚠️ 시작하기 전에 반드시 확인하세요!
Colab 런타임 유형을 **GPU**(T4, L4, A100 등)로 설정해 주셔야 학습이 원활히 진행됩니다.
- 런타임 설정 방법: `런타임` -> `런타임 유형 변경` -> `하드웨어 가속기: GPU` 선택

## 1. GPU 연결 상태 확인

In [ ]:
!nvidia-smi

## 2. 구글 드라이브 마운트
구글 드라이브 바로가기로 연결된 데이터셋(`train_patch.tar` 또는 해제된 폴더)과 체크포인트를 연동하기 위해 구글 드라이브를 연결합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 프로젝트 코드 준비
구글 드라이브에 업로드한 `DLP-D3NET-SIDL` 프로젝트 전체를 Colab 고속 로컬 저장소(`/content`)로 복사합니다.
(구글 드라이브 내에서 직접 실행하는 경우 입출력 병목으로 속도가 많이 느려지므로 로컬 복사를 권장합니다.)

In [ ]:
import os
import shutil

# 구글 드라이브의 프로젝트 폴더 경로 (본인의 드라이브 경로에 맞게 수정해주세요)
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/DLP-D3NET-SIDL'
LOCAL_PROJECT_PATH = '/content/DLP-D3NET-SIDL'

if os.path.exists(DRIVE_PROJECT_PATH):
    print("구글 드라이브에서 프로젝트 폴더를 Colab 로컬로 복사 중...")
    shutil.copytree(DRIVE_PROJECT_PATH, LOCAL_PROJECT_PATH, dirs_exist_ok=True)
    print("프로젝트 코드 복사 완료!")
else:
    print(f"[경고] 구글 드라이브 경로가 존재하지 않습니다: {DRIVE_PROJECT_PATH}")
    print("만약 Colab 세션에 직접 zip 등을 업로드하셨다면 이 단계를 건너뛰어 주세요.")

## 4. 필수 라이브러리 및 패키지 설치

In [ ]:
%cd /content/DLP-D3NET-SIDL/D3Net
!pip install -r requirements.txt

## 5. 데이터셋 준비 (바로가기 활용)
구글 드라이브에 있는 데이터셋 바로가기 형태에 맞춰 아래의 **옵션 A** 또는 **옵션 B** 중 하나만 선택해서 실행하세요.

--- 

### 옵션 A: 드라이브 바로가기가 `train_patch.tar` (압축 파일) 형태인 경우
구글 드라이브 상의 `.tar` 바로가기 파일을 찾아 Colab 로컬 디스크 고속 SSD 영역에 압축 해제합니다.

In [ ]:
# [옵션 A 실행] 바로가기 파일 경로로 지정해 주세요.
TAR_PATH = '/content/drive/MyDrive/train_patch.tar' # 드라이브 내 바로가기 파일 경로로 수정

if os.path.exists(TAR_PATH):
    print(f"[확인] 데이터셋 압축 파일(바로가기) 존재: {TAR_PATH}")
    print("로컬 SSD 영역으로 압축 해제를 시작합니다. (약 5~15분 소요...)")
    !mkdir -p /content/temp_dataset
    !tar -xf {TAR_PATH} -C /content/temp_dataset
    
    # D3Net 학습 폴더로 데이터 이동
    print("데이터를 학습 경로로 이동하는 중...")
    !mkdir -p /content/DLP-D3NET-SIDL/D3Net/data/SIDL
    !mv /content/temp_dataset/home/oem/dataset_final/patch/train /content/DLP-D3NET-SIDL/D3Net/data/SIDL/
    !rm -rf /content/temp_dataset
    print("✨ 데이터셋 배치 완료!")
else:
    print(f"[안내] {TAR_PATH} 경로에 압축 파일이 없거나 옵션 B를 사용할 예정입니다.")

### 옵션 B: 드라이브 바로가기가 이미 압축 해제된 데이터 폴더(`train`) 형태인 경우
이미 폴더 구조가 풀려있는 바로가기라면, 굳이 로컬로 복사하지 않고 심볼릭 링크(Symbolic Link)를 걸어 즉시 연동하거나 로컬로 복사할 수 있습니다.
- 복사 방식: I/O 속도가 빨라 학습이 신속하게 진행되지만 복사 시간이 소요됩니다.
- 심볼릭 링크 방식: 준비 시간 없이 즉시 연결되지만 드라이브 파일 로딩 속도로 인해 학습이 약간 느려질 수 있습니다.

In [ ]:
# [옵션 B 실행] 드라이브 내 데이터 폴더 바로가기 경로 지정
DRIVE_DATA_DIR = '/content/drive/MyDrive/train' # 실제 경로로 수정
TARGET_DATA_DIR = '/content/DLP-D3NET-SIDL/D3Net/data/SIDL/train'

if os.path.exists(DRIVE_DATA_DIR):
    # 학습용 data 디렉토리 생성
    !mkdir -p /content/DLP-D3NET-SIDL/D3Net/data/SIDL
    
    # 방법 1: 고속 학습을 위한 로컬 복사 (권장)
    print("구글 드라이브 폴더 데이터를 Colab 로컬로 복사합니다 (시간이 걸릴 수 있습니다)...")
    !cp -r {DRIVE_DATA_DIR} {TARGET_DATA_DIR}
    print("✨ 로컬 복사 완료!")
    
    # 방법 2: 즉시 연동을 위한 심볼릭 링크 (만약 복사가 오래 걸리면 위 복사 명령을 주석처리하고 아래를 사용하세요)
    # !ln -s {DRIVE_DATA_DIR} {TARGET_DATA_DIR}
    # print("✨ 심볼릭 링크 연결 완료!")
else:
    print(f"[안내] {DRIVE_DATA_DIR} 경로가 존재하지 않거나 옵션 A를 완료하였습니다.")

## 6. D3Net Fine-Tuning 학습 실행
- 런타임 단절 시 체크포인트를 보관할 수 있도록 `--model_folder`를 구글 드라이브 경로로 지정하는 것이 유리합니다.
- Colab GPU 메모리가 부족할 시 `--batch_size 2` 또는 `--img_size 128` 등으로 조절해 주세요.

In [ ]:
%cd /content/DLP-D3NET-SIDL/D3Net

# 학습 실행
!python train_sidl.py \
  --train_dir ./data/SIDL/train \
  --val_dir ./data/SIDL/val \
  --n_epochs 100 \
  --batch_size 4 \
  --img_size 128 \
  --val_img_size 512 \
  --val_batch_size 1 \
  --lr 0.0001 \
  --model_folder /content/drive/MyDrive/DLP-D3NET-SIDL/D3Net/ckpt/sidl_finetune

## 7. 복원 결과 테스트 및 성능 측정
학습된 가중치(`generator_best.pth`)를 불러와 validation 데이터에 적용하고 결과를 시각화합니다.

In [ ]:
!python test.py \
  --image_path ./data/SIDL/val/finger/easy/input \
  --target_data_dir ./data/SIDL/val/finger/easy/target \
  --save_path /content/drive/MyDrive/DLP-D3NET-SIDL/D3Net/images/test_finger_easy \
  --epoch best \
  --model_folder /content/drive/MyDrive/DLP-D3NET-SIDL/D3Net/ckpt/sidl_finetune \
  --img_width 512 \
  --img_height 512